<a href="https://colab.research.google.com/github/Sciform/sciform-rl-and-genai-agents/blob/main/contextual_bandits_genai_agent_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contextual Bandits with an LLM User Simulator — Starter

Companion notebook to `contextual_bandits.ipynb`. Instead of the hand-coded
`get_cost(context, action)` from that notebook, the reward now comes from a
**local LLM agent** that plays the role of the user: given a persona and a
recommended article, the LLM decides whether the user would click.

This starter is intentionally minimal. If you run it as-is you will see the
CB learn *something*, but the click-through rate (CTR) plateaus well below the
rule-based baseline in `contextual_bandits.ipynb`. **Your job is to figure out
why and improve it.** Read the "Your task" section at the bottom before you
start hacking.


## Set up the LLM

We use [Ollama](https://ollama.com) so the whole notebook runs locally (or on
Google Colab) without an API key.

**Local (Windows / macOS / Linux).** Install Ollama once from
https://ollama.com/download — it registers a background service that starts
automatically. Nothing else to do; the next code cell is a no-op for you.

**Google Colab.** Ollama isn't preinstalled. The next code cell detects Colab,
downloads the Ollama binary and launches `ollama serve` in the background.
No runtime restart needed. Prefer a **T4 or L4 GPU runtime** — CPU works but is
noticeably slower per call.

`llama3.2:3b` is roughly 2 GB and returns a JSON click decision in ~100 ms on a
GPU (a few seconds on Colab CPU). Swap in any other Ollama model by changing
`MODEL` in the imports cell.

In [12]:
import os, subprocess, time, urllib.request, urllib.error

# 1. Install dependencies
!sudo apt update && sudo apt install -y pciutils zstd

# 2. Use the official Ollama installation script for proper Linux/Colab support
!curl -fsSL https://ollama.com/install.sh | sh

def _ollama_ready(host="http://127.0.0.1:11434") -> bool:
    try:
        with urllib.request.urlopen(host, timeout=2):
            return True
    except Exception:
        return False

# 3. Clean start: Kill old processes and launch server
!pkill -9 ollama || true

print("Starting Ollama server...")
# Redirect logs to file and run in background
log_file = open("/tmp/ollama.log", "w")
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_ORIGINS": "*"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=log_file, stderr=log_file)

# Wait until responsive
for i in range(20):
    if _ollama_ready():
        print(f"Ollama is ready after {i*2} seconds.")
        break
    time.sleep(2)
else:
    print("Ollama server did not respond. Checking logs:")
    !tail -n 10 /tmp/ollama.log

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
76 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [13]:
%pip install --quiet vowpalwabbit ollama matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 52.7 MB/s eta 0:00:00


In [14]:
import json
import random
from functools import lru_cache
from typing import List, Tuple

import matplotlib.pyplot as plt
import ollama
import pandas as pd
import vowpalwabbit as vw

MODEL = "llama3.2:3b"

# VW minimises cost, so a click is a *negative* cost.
USER_LIKED_ARTICLE = -1.0
USER_DISLIKED_ARTICLE = 0.0

users = ["Tom", "Anna"]
times_of_day = ["morning", "afternoon"]
actions = ["politics", "sports", "music", "food", "finance", "health", "camping"]

In [15]:
# Pull MODEL if it isn't in the local Ollama registry yet.
# Safe to re-run: already-downloaded layers are skipped.
try:
    for chunk in ollama.pull(MODEL, stream=True):
        status = chunk.get("status", "")
        total = chunk.get("total")
        completed = chunk.get("completed")
        if total and completed is not None:
            print(f"\r{status}: {100 * completed / total:5.1f}%", end="")
        elif status:
            print(f"\r{status}", end="")
    print("\ndone")
except ConnectionError as e:
    raise RuntimeError(
        "Cannot reach the Ollama service. Is Ollama running? "
        "Install it from https://ollama.com/download and retry."
    ) from e

success
done


In [16]:
!ollama pull llama3.2

## The LLM reward function

The prompt gives the LLM a persona, the current time of day, and the recommended
article topic, then asks for a strict JSON `{"click": true|false}` answer.

The `(user, time_of_day, action)` space only has 2 × 2 × 7 = 28 combinations, so
we cache decisions with `lru_cache`. The LLM is called at most 28 times over the
whole simulation.


In [17]:
PERSONAS = {
    "Tom": (
        "Tom is a 20-something software developer. He reads political news over "
        "his morning coffee and unwinds with music articles in the afternoon. "
        "He's mostly indifferent to sports, food, finance, health and camping."
    ),
    "Anna": (
        "Anna is a marathon runner. In the morning she catches up on sports news; "
        "in the afternoon she follows politics. Other topics rarely catch her eye."
    ),
}


@lru_cache(maxsize=None)
def _llm_click(user: str, time_of_day: str, action: str) -> bool:
    prompt = (
        f"Persona: {PERSONAS[user]}\n"
        f"Time of day: {time_of_day}\n"
        f'Article topic: "{action}"\n'
        "Would this persona click on this article right now? "
        'Respond with strict JSON of the form {"click": true} or {"click": false}. '
        "No other text."
    )
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"temperature": 0.2},
    )
    try:
        return bool(json.loads(resp["message"]["content"]).get("click", False))
    except json.JSONDecodeError:
        return False


def get_cost_llm(context, action):
    click = _llm_click(context["user"], context["time_of_day"], action)
    return USER_LIKED_ARTICLE if click else USER_DISLIKED_ARTICLE

## Explore what the LLM "wants"

Warm the cache by asking the LLM about every `(user, time_of_day, action)` combo
and print the resulting click matrix. If the LLM has understood the personas,
you should see `1`s concentrated on Tom's morning politics / afternoon music
and Anna's morning sports / afternoon politics.

> **Look at this table carefully.** It's the single best diagnostic for why the
> CB may or may not learn well later — see the "Your task" section below.


In [18]:
rows = []
for u in users:
    for t in times_of_day:
        for a in actions:
            try:
                click_value = int(_llm_click(u, t, a))
                rows.append((u, t, a, click_value))
            except Exception as e:
                print(f"Fehler bei {u}/{t}/{a}: {e}")
                print("Versuche Ollama-Server neu zu starten...")
                # Prozess stoppen
                subprocess.run(["pkill", "-9", "ollama"], check=False)
                time.sleep(3)
                # Cache leeren, falls der Fehler dort gespeichert wurde
                _llm_click.cache_clear()
                # Neustart mit expliziter Host-Umgebung
                log = open("/tmp/ollama.log", "ab")
                subprocess.Popen(
                    ["ollama", "serve"],
                    stdout=log,
                    stderr=log,
                    env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"}
                )
                time.sleep(10) # Längere Wartezeit für den Runner-Start
                # Zweiter Versuch
                try:
                    click_value = int(_llm_click(u, t, a))
                    rows.append((u, t, a, click_value))
                except Exception as retry_e:
                    print(f"Wiederholung fehlgeschlagen: {retry_e}")
                    rows.append((u, t, a, 0))

click_df = pd.DataFrame(rows, columns=["user", "time_of_day", "action", "click"])
click_df.pivot_table(
    index=["user", "time_of_day"], columns="action", values="click"
)

action            camping  finance  food  health  music  politics  sports
user time_of_day                                                         
Anna afternoon        0.0      0.0   0.0     0.0    0.0       1.0     0.0
     morning          0.0      0.0   0.0     0.0    0.0       0.0     1.0
Tom  afternoon        0.0      0.0   0.0     0.0    1.0       0.0     0.0
     morning          0.0      0.0   0.0     0.0    0.0       1.0     0.0

## Reuse the CB machinery

These helpers are copied verbatim from `contextual_bandits.ipynb`. They turn a
CB step into VW's shared+ADF text format, sample from a pmf, and drive a
`Workspace` through predict/learn loops. You should not need to touch them.


In [19]:
def to_vw_example_format(context, actions, cb_label=None):
    if cb_label is not None:
        chosen_action, cost, prob = cb_label
    s = "shared |User user={} time_of_day={}\n".format(
        context["user"], context["time_of_day"]
    )
    for action in actions:
        if cb_label is not None and action == chosen_action:
            s += "0:{}:{} ".format(cost, prob)
        s += "|Action article={} \n".format(action)
    return s[:-1]


def sample_custom_pmf(pmf: List[float]) -> Tuple[int, float]:
    total = sum(pmf)
    pmf = [x / total for x in pmf]
    draw = random.random()
    acc = 0.0
    for i, p in enumerate(pmf):
        acc += p
        if acc > draw:
            return i, p
    return len(pmf) - 1, pmf[-1]


def get_action(workspace, context, actions):
    exs = workspace.parse(
        to_vw_example_format(context, actions),
        labelType=workspace.lContextualBandit,
    )
    try:
        pmf = workspace.predict(exs)
        idx, prob = sample_custom_pmf(pmf)
    finally:
        workspace.finish_example(exs)
    return actions[idx], prob


def run_simulation(workspace, num_iterations, users, times_of_day, actions,
                   cost_function, do_learn=True):
    cost_sum = 0.0
    ctr = []
    for i in range(1, num_iterations + 1):
        context = {
            "user": random.choice(users),
            "time_of_day": random.choice(times_of_day),
        }
        action, prob = get_action(workspace, context, actions)
        cost = cost_function(context, action)
        cost_sum += cost

        if do_learn:
            fmt = to_vw_example_format(context, actions, (action, cost, prob))
            exs = workspace.parse(fmt, labelType=workspace.lContextualBandit)
            try:
                workspace.learn(exs)
            finally:
                workspace.finish_example(exs)

        ctr.append(-cost_sum / i)
    return ctr

## Train the CB: with LLM user vs. without LLM

We train two identical CB learners against two different worlds:

- **with LLM user** — reward comes from `get_cost_llm` (the persona + Ollama agent).
- **without LLM** — reward comes from `get_cost_rule`, a hand-coded rule table
  that matches every Tom/Anna preference exactly (same as `contextual_bandits.ipynb`).

Both learners use `do_learn=True`. The rule-based world has a deterministic
optimal action per context, so its CTR should approach 1.0. The LLM world is
noisier, so its CTR plateau shows how faithfully the LLM roleplays the same
personas.

In [ ]:
def get_cost_rule(context, action):
    if context["user"] == "Tom":
        if context["time_of_day"] == "morning" and action == "politics":
            return USER_LIKED_ARTICLE
        if context["time_of_day"] == "afternoon" and action == "music":
            return USER_LIKED_ARTICLE
    elif context["user"] == "Anna":
        if context["time_of_day"] == "morning" and action == "sports":
            return USER_LIKED_ARTICLE
        if context["time_of_day"] == "afternoon" and action == "politics":
            return USER_LIKED_ARTICLE
    return USER_DISLIKED_ARTICLE


random.seed(0)
num_iterations = 3000

ws_llm = vw.Workspace("--cb_explore_adf --interactions=UA --epsilon=0.2 --quiet")
ctr_llm = run_simulation(
    ws_llm, num_iterations, users, times_of_day, actions, get_cost_llm
)

ws_rule = vw.Workspace("--cb_explore_adf --interactions=UA --epsilon=0.2 --quiet")
ctr_rule = run_simulation(
    ws_rule, num_iterations, users, times_of_day, actions, get_cost_rule
)

print(f"CTR with LLM user:        {ctr_llm[-1]:.3f}")
print(f"CTR without LLM (rules):  {ctr_rule[-1]:.3f}")

## Visualise the CTR curves


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, num_iterations + 1), ctr_llm, label="with LLM user")
plt.plot(
    range(1, num_iterations + 1), ctr_rule,
    label="without LLM (hand-coded rules)", alpha=0.7,
)
plt.xlabel("iteration")
plt.ylabel("CTR")
plt.ylim([0, 1])
plt.legend()
plt.title("Contextual bandit: LLM user simulator vs. rule-based reward")
plt.show()

## Your task

Compare the CTR curve above with the rule-based scenario in
`contextual_bandits.ipynb` (which plateaus near ~0.85). The LLM simulator here
usually plateaus much lower. Your job is to diagnose why and close the gap.

### Diagnose first

1. Re-read the click matrix from the "Explore what the LLM wants" cell.
   For each `(user, time_of_day)` context, does the LLM click on **exactly** the
   article the persona should like? Missing clicks on the "right" article cap
   the CTR from above; extra clicks on the "wrong" articles slow learning.
2. Compute an upper bound on CTR: what fraction of the 4 canonical
   `(user, time_of_day)` contexts have their expected article marked `1` in the
   matrix? That's the best the CB can ever do against this LLM.

### Improvements to try (pick at least two)

- **Prompt engineering.** Rewrite the prompt inside `_llm_click` to be more
  decisive. Explicitly tell the LLM to click when the topic clearly matches
  the persona's interests at this time of day, and not to be cautious.
- **Few-shot examples.** Add one or two `Persona → click?` examples (using an
  *unrelated* persona so you don't leak the answer) directly inside the prompt.
- **Sharper personas.** Rewrite `PERSONAS["Tom"]` and `PERSONAS["Anna"]` so the
  preferences are unambiguous. Does adding "she always clicks morning sports
  articles" help?
- **Graded reward.** Instead of returning `True`/`False`, ask the LLM for a
  probability in `[0.0, 1.0]` (`{"probability": 0.9}`) and use `-p` as the VW
  cost. VW handles continuous costs natively.
- **Bigger model.** Pull `llama3.1:8b` or `qwen2.5:7b-instruct`, set
  `MODEL = "..."`, clear the cache with `_llm_click.cache_clear()`, and rerun.
- **Persona drift.** After 1 500 iterations, change `PERSONAS["Tom"]` (e.g.
  add "recently got into cooking"), clear the cache, and continue training.
  Does VW recover? How fast?

### Deliverable

Produce a plot with three curves in the same axes:
1. Original starter (this notebook, unchanged).
2. Your best improved version.
3. The rule-based reference from `contextual_bandits.ipynb`.

Report your final CTR after 3 000 iterations and 200 words on what worked.
